   
# Raw Data to Delta Tables

## Purpose

This notebook converts raw datasets from the volume into Delta tables in Unity Catalog.

**Simple workflow**: Load dataset → Save to Delta with `_raw` suffix

## What this notebook does:

1. **Load each raw dataset from volume**
2. **Save each to its own Delta table** with `_raw` suffix
3. **No transformations, joins, or calculations**

## Output Tables:

| Dataset | Source File | Delta Table |
|---------|-------------|-------------|
| Provincial population | `padron/padron_2024.csv` | `padron_raw` |
| Municipal population | `padron/padron_municipal_2024.csv` | `padron_municipal_raw` |
| Municipal geometries | `spain_unidades_administrativas/*.gml` | `municipios_geo_raw` |
| Province geometries | `spain_unidades_administrativas/*.gml` | `provincias_geo_raw` |
| CCAA geometries | `spain_unidades_administrativas/*.gml` | `ccaa_geo_raw` |

**Note**: Tables with `geo_raw` suffix contain geometry information from IGN GML files.

---

**Project**: P-01 Phase A - Where does Spain actually live?  
**Notebook**: 2 of N (raw data → Delta tables)  
**Last Updated**: May 30, 2026

In [0]:
%run ./0_setup

In [0]:
# Unity Catalog configuration
UC_CATALOG = "geospatial"
UC_SCHEMA = "spain_population_analysis"
VOLUME_PATH = Path('/Volumes/geospatial/spain_population_analysis/datasets')

# Input paths from volume
PADRON_PATH = VOLUME_PATH / 'padron' / 'padron_2024.csv'
PADRON_MUNICIPAL_PATH = VOLUME_PATH / 'padron' / 'poblacion_sexo_municipios_edad.csv'  # Complete municipal dataset
MUNICIPAL_GML_PATH = VOLUME_PATH / 'spain_unidades_administrativas' / 'au_AdministrativeUnit_4thOrder0.gml'
PROVINCE_GML_PATH = VOLUME_PATH / 'spain_unidades_administrativas' / 'au_AdministrativeUnit_3rdOrder0.gml'
CCAA_GML_PATH = VOLUME_PATH / 'spain_unidades_administrativas' / 'au_AdministrativeUnit_2ndOrder0.gml'

print("Configuration:")
print(f"  Catalog.Schema: {UC_CATALOG}.{UC_SCHEMA}")
print(f"  Volume: {VOLUME_PATH}")
print(f"\nInput Files:")
print(f"  ✓ Padrón (Provincial): {PADRON_PATH.name}")
print(f"  ✓ Padrón (Municipal - ALL): {PADRON_MUNICIPAL_PATH.name}")
print(f"  ✓ Municipal GML: {MUNICIPAL_GML_PATH.name}")
print(f"  ✓ Province GML: {PROVINCE_GML_PATH.name}")
print(f"  ✓ CCAA GML: {CCAA_GML_PATH.name}")
print(f"\nOutput: Raw Delta tables with '_raw' suffix")
print("\n✓ Configuration complete")

In [0]:
# Load raw INE Padrón population data from volume
print("Loading INE Padrón provincial population data...\n")

df_padron_raw = pd.read_csv(PADRON_PATH, sep=';', encoding='utf-8')

print(f"✓ Loaded {len(df_padron_raw):,} rows")
print(f"\nOriginal columns: {list(df_padron_raw.columns)}")
print(f"\nFirst few rows (raw):")
display(df_padron_raw.head(5))

# Clean and structure the data
print("\nCleaning and structuring data...")

# Filter out 'Total' row (national aggregate) and rows with missing data
df_padron = df_padron_raw[
    (df_padron_raw['Provincias'] != 'Total') & 
    (df_padron_raw['Total'].notna())
].copy()

# Split 'Provincias' column into codigo and provincia
df_padron['codigo'] = df_padron['Provincias'].str[:2].astype(int)
df_padron['provincia'] = df_padron['Provincias'].str[3:].str.strip()

# Rename and fix data types
df_padron['sexo'] = df_padron['Sexo']
df_padron['periodo'] = df_padron['Periodo'].astype(int)

# Convert 'Total' from string (with periods) to int
df_padron['total'] = df_padron['Total'].str.replace('.', '', regex=False).astype(int)

# Select final columns in desired order
df_padron = df_padron[['codigo', 'provincia', 'sexo', 'periodo', 'total']]

print(f"\n✓ Cleaned data structure:")
print(f"  Columns: {list(df_padron.columns)}")
print(f"  Types: {dict(df_padron.dtypes)}")
print(f"\nFirst few rows (cleaned):")
display(df_padron.head(10))

In [0]:
# Load municipal boundaries from GML
print("Loading municipal administrative boundaries...")
print("Note: Large file (137 MB), may take 30-60 seconds...\n")

gdf_municipios = gpd.read_file(MUNICIPAL_GML_PATH)

print(f"✓ Loaded {len(gdf_municipios):,} municipalities")
print(f"  CRS: {gdf_municipios.crs}")
print(f"  Columns: {len(gdf_municipios.columns)}")
display(gdf_municipios.head(3))

In [0]:
# Save to Delta table: padron_raw
from pyspark.sql.functions import expr

print(f"Saving to Delta: {UC_CATALOG}.{UC_SCHEMA}.padron_raw\n")

# Convert to Spark DataFrame
spark_padron = spark.createDataFrame(df_padron)

# Write to Delta table (overwriteSchema allows schema changes)
spark_padron.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{UC_CATALOG}.{UC_SCHEMA}.padron_raw"
)

print(f"✓ Saved {len(df_padron):,} rows to padron_raw")
print(f"\nVerify:")
spark.sql(f"""
    SELECT COUNT(*) as total_rows,
           COUNT(DISTINCT codigo) as unique_provincias,
           MIN(periodo) as min_year,
           MAX(periodo) as max_year
    FROM {UC_CATALOG}.{UC_SCHEMA}.padron_raw
""").show()

In [0]:
# Load raw INE municipal population data from volume (ALL municipalities, ~8,100+)
# This is a large file with millions of rows
print("Loading INE municipal population data (ALL municipalities)...")
print("Note: Large file, may take a moment...\n")

df_padron_municipal_raw = pd.read_csv(PADRON_MUNICIPAL_PATH, sep=';', encoding='utf-8', dtype=str)

print(f"✓ Loaded {len(df_padron_municipal_raw):,} rows")
print(f"\nOriginal columns: {list(df_padron_municipal_raw.columns)}")
print(f"\nFirst few rows (raw):")
display(df_padron_municipal_raw.head(5))

# Rename columns to standardized names (keep all as strings)
# Remove parentheses and special characters from column names (Delta requirement)
print("\nRenaming columns...")

df_padron_municipal = df_padron_municipal_raw.rename(columns={
    'Sexo': 'sexo',
    'Municipios': 'municipio',
    'Edad (grupos quinquenales)': 'grupo_edad',
    'Periodo': 'periodo',
    'Total': 'total'
})

print(f"\n✓ Renamed columns:")
print(f"  Columns: {list(df_padron_municipal.columns)}")
print(f"  All columns kept as strings (no parsing)")
print(f"  Total rows: {len(df_padron_municipal):,}")

print(f"\nFirst few rows (renamed):")
display(df_padron_municipal.head(10))

In [0]:
# Save to Delta table: padron_municipal_raw
print(f"Saving to Delta: {UC_CATALOG}.{UC_SCHEMA}.padron_municipal_raw\n")

# Convert to Spark DataFrame
spark_padron_municipal = spark.createDataFrame(df_padron_municipal)

# Write to Delta table (overwriteSchema allows schema changes)
spark_padron_municipal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{UC_CATALOG}.{UC_SCHEMA}.padron_municipal_raw"
)

print(f"✓ Saved {len(df_padron_municipal):,} rows to padron_municipal_raw")
print(f"\n{'='*60}")
print("DATASET SUMMARY")
print(f"{'='*60}")

# Comprehensive verification with multiple dimensions
stats = spark.sql(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT SUBSTR(municipio, 1, 5)) as unique_municipios,
        COUNT(DISTINCT sexo) as unique_sexo,
        COUNT(DISTINCT grupo_edad) as unique_grupos_edad,
        COUNT(DISTINCT periodo) as unique_periodos,
        MIN(CAST(SUBSTR(periodo, -4) AS INT)) as min_year,
        MAX(CAST(SUBSTR(periodo, -4) AS INT)) as max_year
    FROM {UC_CATALOG}.{UC_SCHEMA}.padron_municipal_raw
""").collect()[0]

print(f"\nData Coverage:")
print(f"  Total rows: {stats['total_rows']:,}")
print(f"  Unique municipalities: {stats['unique_municipios']:,}")
print(f"  Year range: {stats['min_year']} - {stats['max_year']} ({stats['unique_periodos']} periods)")

print(f"\nDimensions:")
print(f"  Sex categories: {stats['unique_sexo']}")
print(f"  Age groups: {stats['unique_grupos_edad']}")

print(f"\nBreakdown by category:")

# Sex breakdown
print(f"\n  Sex:")
spark.sql(f"""
    SELECT sexo, COUNT(*) as count, 
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM {UC_CATALOG}.{UC_SCHEMA}.padron_municipal_raw
    GROUP BY sexo
    ORDER BY count DESC
""").show(truncate=False)

# Age group breakdown
print(f"  Age groups (top 10):")
spark.sql(f"""
    SELECT grupo_edad, COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM {UC_CATALOG}.{UC_SCHEMA}.padron_municipal_raw
    GROUP BY grupo_edad
    ORDER BY count DESC
    LIMIT 10
""").show(truncate=False)

# Sample of unique municipalities
print(f"  Sample municipalities:")
spark.sql(f"""
    SELECT DISTINCT municipio
    FROM {UC_CATALOG}.{UC_SCHEMA}.padron_municipal_raw
    WHERE municipio NOT LIKE 'Total%'
    ORDER BY municipio
    LIMIT 10
""").show(truncate=False)

print(f"\n{'='*60}")
print("✓ Municipal population data successfully saved to Delta")
print(f"{'='*60}")

In [0]:
# Load province boundaries from GML
print("Loading province administrative boundaries...\n")

gdf_provincias = gpd.read_file(PROVINCE_GML_PATH)

print(f"✓ Loaded {len(gdf_provincias):,} provinces")
print(f"  CRS: {gdf_provincias.crs}")
print(f"  Columns: {len(gdf_provincias.columns)}")
display(gdf_provincias.head(3))

In [0]:
# Save to Delta table: municipios_geo_raw
from pyspark.sql.functions import expr

print(f"Saving to Delta: {UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw\n")

# Convert geometry to WKT for Spark
gdf_municipios['geometry_wkt'] = gdf_municipios.geometry.to_wkt()

# Keep all original columns, but avoid duplicate
cols_to_keep = [col for col in gdf_municipios.columns if col != 'geometry']  # do NOT append ['geometry_wkt']
municipal_output = gdf_municipios[cols_to_keep].copy()

# Convert to Spark DataFrame
spark_municipal = spark.createDataFrame(municipal_output)

# Convert WKT to geometry
spark_municipal = spark_municipal.withColumn(
    "geometry",
    expr("st_geomfromwkt(geometry_wkt)")
).drop("geometry_wkt")

# Write to Delta
spark_municipal.write.format("delta").mode("overwrite").saveAsTable(
    f"{UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw"
)

print(f"✓ Saved {len(municipal_output):,} municipalities to municipios_geo_raw")
spark.sql(f"SELECT COUNT(*) as count FROM {UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw").show()

In [0]:
# Save to Delta table: provincias_geo_raw
print(f"Saving to Delta: {UC_CATALOG}.{UC_SCHEMA}.provincias_geo_raw\n")

# Convert geometry to WKT
gdf_provincias['geometry_wkt'] = gdf_provincias.geometry.to_wkt()

# Keep all original columns, but avoid duplicate
cols_to_keep = [col for col in gdf_provincias.columns if col != 'geometry']  # do NOT append ['geometry_wkt']
province_output = gdf_provincias[cols_to_keep].copy()

# Convert to Spark DataFrame
spark_province = spark.createDataFrame(province_output)

# Convert WKT to geometry
spark_province = spark_province.withColumn(
    "geometry",
    expr("st_geomfromwkt(geometry_wkt)")
).drop("geometry_wkt")

# Write to Delta
spark_province.write.format("delta").mode("overwrite").saveAsTable(
    f"{UC_CATALOG}.{UC_SCHEMA}.provincias_geo_raw"
)

print(f"✓ Saved {len(province_output)} provinces to provincias_geo_raw")
spark.sql(f"SELECT COUNT(*) as count FROM {UC_CATALOG}.{UC_SCHEMA}.provincias_geo_raw").show()

In [0]:
# Load CCAA (Autonomous Community) boundaries from GML
print("Loading CCAA administrative boundaries...\n")

gdf_ccaa = gpd.read_file(CCAA_GML_PATH)

print(f"✓ Loaded {len(gdf_ccaa)} autonomous communities")
print(f"  CRS: {gdf_ccaa.crs}")
print(f"  Columns: {len(gdf_ccaa.columns)}")
display(gdf_ccaa.head(3))

In [0]:
# Save to Delta table: ccaa_geo_raw
print(f"Saving to Delta: {UC_CATALOG}.{UC_SCHEMA}.ccaa_geo_raw\n")

# Convert geometry to WKT
gdf_ccaa['geometry_wkt'] = gdf_ccaa.geometry.to_wkt()

# Keep all original columns, but avoid duplicate
cols_to_keep = [col for col in gdf_ccaa.columns if col != 'geometry']  # do NOT append ['geometry_wkt']
ccaa_output = gdf_ccaa[cols_to_keep].copy()

# Convert to Spark DataFrame
spark_ccaa = spark.createDataFrame(ccaa_output)

# Convert WKT to geometry
spark_ccaa = spark_ccaa.withColumn(
    "geometry",
    expr("st_geomfromwkt(geometry_wkt)")
).drop("geometry_wkt")

# Write to Delta
spark_ccaa.write.format("delta").mode("overwrite").saveAsTable(
    f"{UC_CATALOG}.{UC_SCHEMA}.ccaa_geo_raw"
)

print(f"✓ Saved {len(ccaa_output)} autonomous communities to ccaa_geo_raw")
spark.sql(f"SELECT COUNT(*) as count FROM {UC_CATALOG}.{UC_SCHEMA}.ccaa_geo_raw").show()

In [0]:
# EDA: Summary of municipios_geo_raw
print(f"{'='*60}")
print("EXPLORATORY DATA ANALYSIS: municipios_geo_raw")
print(f"{'='*60}\n")

# Basic table info
print("Table Information:")
print(f"  Catalog: {UC_CATALOG}")
print(f"  Schema: {UC_SCHEMA}")
print(f"  Table: municipios_geo_raw\n")

# Row count
total_rows = spark.sql(f"SELECT COUNT(*) as count FROM {UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw").collect()[0]['count']
print(f"Total Municipalities: {total_rows:,}\n")

# Schema
print("Table Schema:")
spark.sql(f"DESCRIBE {UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw").show(truncate=False)

# Sample rows
print("\nSample Rows (first 5):")
spark.sql(f"""
    SELECT nationalCode, localId, text as municipality_name, 
           ST_AsText(geometry) as geometry_preview
    FROM {UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw
    LIMIT 5
""").show(truncate=50)

# Summary statistics
print("\nSummary Statistics:")
spark.sql(f"""
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT nationalCode) as unique_national_codes,
        COUNT(DISTINCT localId) as unique_local_ids,
        COUNT(DISTINCT text) as unique_municipality_names,
        COUNT(geometry) as geometries_present,
        SUM(CASE WHEN geometry IS NULL THEN 1 ELSE 0 END) as missing_geometries
    FROM {UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw
""").show(truncate=False)

# Sample municipality names
print("\nSample Municipality Names (A-Z):")
spark.sql(f"""
    SELECT text as municipality_name, nationalCode
    FROM {UC_CATALOG}.{UC_SCHEMA}.municipios_geo_raw
    WHERE text IS NOT NULL
    ORDER BY text
    LIMIT 10
""").show(truncate=False)

print(f"\n{'='*60}")
print("✓ EDA Complete")
print(f"{'='*60}")

   
## ✅ Raw Delta Tables Created

The following raw Delta tables have been created:

| Table Name | Description | Row Count |
|------------|-------------|----------|
| `geospatial.spain_population_analysis.padron_raw` | INE Padrón provincial population | ~4,000 rows |
| `geospatial.spain_population_analysis.padron_municipal_raw` | INE Padrón municipal population | ~4,000 rows |
| `geospatial.spain_population_analysis.municipios_geo_raw` | Municipal boundaries (GML) | ~8,220 |
| `geospatial.spain_population_analysis.provincias_geo_raw` | Province boundaries (GML) | 53 |
| `geospatial.spain_population_analysis.ccaa_geo_raw` | CCAA boundaries (GML) | ~19 |

**All tables contain raw data with no transformations applied.**

**Note**: Tables with `geo_raw` suffix contain geometry information from IGN GML files.

---

### Next Steps:

Notebook 3 will handle data transformations:
* Extract INE códigos from INSPIRE codes
* Join population to geometries
* Calculate density
* Create analysis-ready tables